In [53]:
import os, sys
from pathlib import Path
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [54]:
import os
from pathlib import Path

# ---------- 1) Stop any existing SparkSession (if any) ----------
try:
    spark.stop()
    print("Stopped existing SparkSession.")
except Exception:
    print("No SparkSession to stop (ok).")

# ---------- 2) Clear PySpark global state (fixes dead Py4J gateway) ----------
try:
    from pyspark.context import SparkContext
    from pyspark.sql import SparkSession

    for k in ["PYSPARK_GATEWAY_PORT", "PYSPARK_GATEWAY_SECRET"]:
        if k in os.environ:
            os.environ.pop(k, None)
            print(f"Cleared env {k}")

    SparkContext._active_spark_context = None
    SparkContext._gateway = None
    SparkContext._jvm = None

    try:
        SparkSession._instantiatedContext = None
        SparkSession._activeSession = None
        SparkSession._defaultSession = None
    except Exception:
        pass

    print("✅ PySpark internals reset")
except Exception as e:
    print("⚠️ Reset step warning:", type(e).__name__, e)

# ---------- 3) Start Spark (stable local mode) ----------
from pyspark.sql import SparkSession

OUTPUT_ROOT = Path.home() / "opendota_processed"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SPARK_LOCAL_DIR = OUTPUT_ROOT / "spark_local"
SPARK_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("OpenDota_Notebook")
    .master("local[*]")  # stable; we can change later
    .config("spark.local.dir", str(SPARK_LOCAL_DIR))
    .config("spark.sql.shuffle.partitions", "400")
    .config("spark.sql.adaptive.enabled", "true")
    # IMPORTANT: avoid surprise broadcasts that can destabilize the JVM
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("✅ Spark started:", spark.version)
print(" - Master:", spark.sparkContext.master)
print(" - Spark UI:", spark.sparkContext.uiWebUrl)
print("alive_check =", spark.range(1).count())

Stopped existing SparkSession.
✅ PySpark internals reset


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/14 10:56:04 WARN Utils: Your hostname, Chas-MacBook-Air-3.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.7 instead (on interface en0)
26/02/14 10:56:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/14 10:56:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/14 10:56:05 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


✅ Spark started: 4.1.1
 - Master: local[*]
 - Spark UI: http://172.20.10.7:4040
alive_check = 1


In [55]:
import os
from pathlib import Path

# 1) Stop if possible
try:
    spark.stop()
    print("Stopped SparkSession.")
except Exception:
    print("No SparkSession to stop (ok).")

# 2) Hard reset PySpark globals (dead gateway fix)
from pyspark.context import SparkContext
from pyspark.sql import SparkSession

for k in ["PYSPARK_GATEWAY_PORT", "PYSPARK_GATEWAY_SECRET"]:
    os.environ.pop(k, None)

SparkContext._active_spark_context = None
SparkContext._gateway = None
SparkContext._jvm = None
SparkSession._instantiatedContext = None
SparkSession._activeSession = None
SparkSession._defaultSession = None

# 3) Start Spark (stable settings)
OUTPUT_ROOT = Path.home() / "opendota_processed"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SPARK_LOCAL_DIR = OUTPUT_ROOT / "spark_local"
SPARK_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("OpenDota_Preprocessing_Notebook_Stable")
    .master("local[4]")                 # <-- reduce threads for stability
    .config("spark.ui.enabled", "true")# <-- remove UI port issues while debugging
    .config("spark.local.dir", str(SPARK_LOCAL_DIR))
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.sql.adaptive.enabled", "false")  # <-- disable AQE for now
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark started:", spark.version)
print(" - Master:", spark.sparkContext.master)
print(" - Spark UI:", spark.sparkContext.uiWebUrl)
print("alive_check =", spark.range(1).count())

Stopped SparkSession.


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/14 10:56:10 WARN Utils: Your hostname, Chas-MacBook-Air-3.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.7 instead (on interface en0)
26/02/14 10:56:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/14 10:56:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/14 10:56:10 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


✅ Spark started: 4.1.1
 - Master: local[4]
 - Spark UI: http://172.20.10.7:4040
alive_check = 1


In [58]:
input_path = "opendota_processed/matches_parts/part-*.csv"

In [ ]:
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, StringType

# Th
schema = StructType([
    StructField("match_id", LongType(), True),
    StructField("match_seq_num", LongType(), True), # Needed to maintain index position
    StructField("radiant_win", StringType(), True),
    StructField("start_time", LongType(), True),
    StructField("duration", IntegerType(), True)
])

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load(input_path)

df_matches = df_raw.select("match_id", "radiant_win", "start_time", "duration")

26/02/14 10:57:54 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: opendota_processed/matches_parts/part-*.csv.
java.io.FileNotFoundException: File opendota_processed/matches_parts/part-*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	a

In [60]:
df_matches.select("match_id","radiant_win", "start_time", "duration").show(5)

+---------+-----------+----------+--------+
| match_id|radiant_win|start_time|duration|
+---------+-----------+----------+--------+
|473260719|          t|1390019262|    3234|
|473260722|          f|1390019274|    2360|
|473260723|          f|1390019260|    2546|
|473260724|          f|1390019278|    2647|
|473260725|          t|1390019250|    3030|
+---------+-----------+----------+--------+
only showing top 5 rows


In [ ]:
from pyspark.sql import functions as F

# count of matches before drop
before_count = df_matches.count()

# constructing clean match data
matches_clean = df_matches.withColumn("match_id", F.col("match_id").cast("long")) \
                          .withColumn("start_time", F.col("start_time").cast("long")) \
                          .withColumn("duration", F.col("duration").cast("int")) \
                          .withColumn("radiant_win", F.col("radiant_win") == "t") \
                          .dropna(subset=["match_id"]) \
                          .dropDuplicates(["match_id"])

# Justification: Matches < 5 mins (300s) are usually 'null' games (early abandons).
# Matches > 3 hours (10800s) are extreme outliers/server errors in Dota 2.
lower_bound = 300
upper_bound = 10800
matches_clean = matches_clean.filter((F.col("duration") >= lower_bound) & (F.col("duration") <= upper_bound))

# D) Capture After Count
after_count = matches_clean.count()

In [ ]:
# null rate calc
null_rate_df = matches_clean.select([
    (F.count(F.when(F.col(c).isNull(), c)) / after_count * 100).alias(c) 
    for c in matches_clean.columns
])

print("--- TASK 4: EVIDENCE ---")
print(f"Before cleaning: {before_count} rows")
print(f"After cleaning:  {after_count} rows")
print(f"Rows removed:    {before_count - after_count}")

print("\n--- Null Rate Summary (%) ---")
null_rate_df.show()

--- TASK 4: EVIDENCE ---
Before cleaning: 500000 rows
After cleaning:  497320 rows
Rows removed:    2680

--- Null Rate Summary (%) ---


+--------+--------------------+----------+--------+
|match_id|         radiant_win|start_time|duration|
+--------+--------------------+----------+--------+
|     0.0|2.010777768840987...|       0.0|     0.0|
+--------+--------------------+----------+--------+



In [63]:
matches_clean.printSchema()
matches_clean.limit(5).show()

root
 |-- match_id: long (nullable = true)
 |-- radiant_win: boolean (nullable = true)
 |-- start_time: long (nullable = true)
 |-- duration: integer (nullable = true)



+----------+-----------+----------+--------+
|  match_id|radiant_win|start_time|duration|
+----------+-----------+----------+--------+
| 473260800|      false|1390019708|    1843|
|1009842190|      false|1415362477|    2950|
|1808667856|      false|1442797852|    3096|
| 692810562|      false|1401589879|    2141|
|1980325600|       true|1449295127|    2140|
+----------+-----------+----------+--------+



In [ ]:
from pyspark.sql import functions as F

# derive day, year, month, and duration_min
matches_features = matches_clean.withColumn("start_ts", F.from_unixtime("start_time").cast("timestamp")) \
    .withColumn("year", F.year("start_ts")) \
    .withColumn("month", F.month("start_ts")) \
    .withColumn("day", F.day("start_ts")) \
    .withColumn("duration_min", F.col("duration") / 60)

# bins: <20, 20–30, 30-40, 40-50, 50-60, 60+
matches_features = matches_features.withColumn("duration_bin", 
    F.when(F.col("duration_min") < 20, "<20")
     .when((F.col("duration_min") >= 20) & (F.col("duration_min") < 30), "20-30")
     .when((F.col("duration_min") >= 30) & (F.col("duration_min") < 40), "30-40")
     .when((F.col("duration_min") >= 40) & (F.col("duration_min") < 50), "40-50")
     .when((F.col("duration_min") >= 50) & (F.col("duration_min") < 60), "50-60")
     .otherwise("60+")
)

# preview of newest features 
matches_features.select("match_id", "start_ts", "year", "month", "day", "duration_min", "duration_bin").show(5)

+----------+-------------------+----+-----+---+------------------+------------+
|  match_id|           start_ts|year|month|day|      duration_min|duration_bin|
+----------+-------------------+----+-----+---+------------------+------------+
| 473260800|2014-01-18 12:35:08|2014|    1| 18|30.716666666666665|       30-40|
|1009842190|2014-11-07 20:14:37|2014|   11|  7|49.166666666666664|       40-50|
|1808667856|2015-09-21 09:10:52|2015|    9| 21|              51.6|       50-60|
| 692810562|2014-06-01 10:31:19|2014|    6|  1| 35.68333333333333|       30-40|
|1980325600|2015-12-05 13:58:47|2015|   12|  5|35.666666666666664|       30-40|
+----------+-------------------+----+-----+---+------------------+------------+
only showing top 5 rows


In [ ]:
# define the output path
parquet_path = f"data/matches_features.parquet"

# write the data partitioned by year and month
matches_features.write.mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet(parquet_path)

print(f"✅ Data successfully written to: {parquet_path}")

✅ Data successfully written to: data/matches_features.parquet


In [78]:
import os
os.system(f"ls -R {parquet_path} | head -n 20")

_SUCCESS
year=2011
year=2012
year=2013
year=2014
year=2015
year=2016

data/matches_features.parquet/year=2011:
month=10
month=11
month=12
month=7
month=9

data/matches_features.parquet/year=2011/month=10:
part-00000-dba8c8b4-bb6c-4744-b53d-54a28e7d1587.c000.snappy.parquet
part-00001-dba8c8b4-bb6c-4744-b53d-54a28e7d1587.c000.snappy.parquet
part-00002-dba8c8b4-bb6c-4744-b53d-54a28e7d1587.c000.snappy.parquet
part-00006-dba8c8b4-bb6c-4744-b53d-54a28e7d1587.c000.snappy.parquet


ls: stdout: Undefined error: 0


0

In [79]:
df_p = spark.read.parquet(parquet_path)
filtered_query = df_p.filter((F.col("year") == 2014) & (F.col("month") == 1))
filtered_query.explain(True)

== Parsed Logical Plan ==
'Filter 'and('`=`('year, 2014), '`=`('month, 1))
+- Relation [match_id#547L,radiant_win#548,start_time#549L,duration#550,start_ts#551,duration_min#552,duration_bin#553,year#554,month#555] parquet

== Analyzed Logical Plan ==
match_id: bigint, radiant_win: boolean, start_time: bigint, duration: int, start_ts: timestamp, duration_min: double, duration_bin: string, year: int, month: int
Filter ((year#554 = 2014) AND (month#555 = 1))
+- Relation [match_id#547L,radiant_win#548,start_time#549L,duration#550,start_ts#551,duration_min#552,duration_bin#553,year#554,month#555] parquet

== Optimized Logical Plan ==
Filter ((isnotnull(year#554) AND isnotnull(month#555)) AND ((year#554 = 2014) AND (month#555 = 1)))
+- Relation [match_id#547L,radiant_win#548,start_time#549L,duration#550,start_ts#551,duration_min#552,duration_bin#553,year#554,month#555] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [match_id#547L,radiant_win#548,start_time#549L,duration#

In [ ]:
# aggregate by year and month
# calculate both count (activity) and mean (trend)
activity_trend = df_p.groupBy("year", "month") \
    .agg(
        F.count("match_id").alias("match_count"),
        F.avg("duration_min").alias("avg_duration")
    ) \
    .orderBy("year", "month") \
    .toPandas() # small result set, safe for Pandas

print("--- Match Activity & Duration Trends (First 5 Rows) ---")
print(activity_trend.head())

--- Match Activity & Duration Trends (First 5 Rows) ---
   year  month  match_count  avg_duration
0  2011      7            7     48.802381
1  2011      9            2     18.216667
2  2011     10           19     41.304386
3  2011     11           34     37.098529
4  2011     12           96     40.998438


In [72]:
# Count matches per bin
duration_dist = df_p.groupBy("duration_bin") \
    .count() \
    .orderBy("duration_bin") \
    .toPandas()

print("\n--- Duration Distribution ---")
print(duration_dist)


--- Duration Distribution ---
  duration_bin   count
0        20-30   56290
1        30-40  162962
2        40-50  157166
3        50-60   78304
4          60+   29296
5          <20   13302


In [73]:
# Calculate win rate percentage by year
win_rate_stability = df_p.withColumn("win_int", F.col("radiant_win").cast("int")) \
    .groupBy("year") \
    .agg(F.avg("win_int").alias("radiant_win_rate")) \
    .orderBy("year") \
    .toPandas()

print("\n--- Radiant Win Rate Stability ---")
print(win_rate_stability)


--- Radiant Win Rate Stability ---
   year  radiant_win_rate
0  2011          0.563291
1  2012          0.509705
2  2013          0.515877
3  2014          0.519029
4  2015          0.515306
5  2016          0.513675


In [75]:
# Convert start_ts to a simple date format and count
daily_activity = df_p.withColumn("date", F.to_date("start_ts")) \
    .groupBy("date") \
    .count() \
    .withColumnRenamed("count", "total_games") \
    .orderBy("date") \
    .toPandas()

print("--- Daily Match Activity ---")
print(daily_activity.head(10))

--- Daily Match Activity ---
         date  total_games
0  2011-07-02            1
1  2011-07-03            1
2  2011-07-04            1
3  2011-07-07            1
4  2011-07-09            1
5  2011-07-17            1
6  2011-07-24            1
7  2011-09-26            1
8  2011-09-30            1
9  2011-10-01            2


In [ ]:
play_time_dist = df_p.groupBy("duration_bin") \
    .count() \
    .withColumnRenamed("count", "frequency") \
    .orderBy("duration_bin") \
    .toPandas()

print("\n--- Play Time Distribution ---")
print(play_time_dist)


--- Play Time Distribution ---
  duration_bin  frequency
0        20-30      56290
1        30-40     162962
2        40-50     157166
3        50-60      78304
4          60+      29296
5          <20      13302


In [77]:
# Calculate mean of radiant_win (True=1, False=0)
win_rate_trend = df_p.withColumn("win_val", F.col("radiant_win").cast("int")) \
    .groupBy("year", "month") \
    .agg(F.avg("win_val").alias("radiant_win_rate")) \
    .orderBy("year", "month") \
    .toPandas()

print("\n--- Win Rate Trend (Monthly) ---")
print(win_rate_trend.head(10))


--- Win Rate Trend (Monthly) ---
   year  month  radiant_win_rate
0  2011      7          0.142857
1  2011      9          1.000000
2  2011     10          0.684211
3  2011     11          0.441176
4  2011     12          0.604167
5  2012      1          0.476190
6  2012      2          0.514360
7  2012      3          0.518519
8  2012      4          0.559160
9  2012      5          0.534247
